# Unified Video Analysis Pipeline (Colab)

Runs the Streamlit app from **https://github.com/SudeepIITM/MicroVLM_Edge1** via a Cloudflare tunnel.

**Before running:** set the runtime to GPU -> *Runtime > Change runtime type > Hardware accelerator > GPU (T4)*.

Run the cells top-to-bottom. The last cell prints a public `https://*.trycloudflare.com` URL.

Pipeline components:
1. **Summary** - Qwen3-VL + Q-Former LoRA checkpoint
2. **Multi-class** - EnhancedTemporalAdapterModel using the generated summary
3. **Binary** - SimpleAdapter + LoRA ensemble

## 1. Clone the repository

In [ ]:
import os, sys, subprocess, shutil, time

REPO_URL = "https://github.com/SudeepIITM/MicroVLM_Edge1.git"
REPO_DIR = "/content/MicroVLM_Edge1"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "pull"], cwd=REPO_DIR, check=True)

print(f"Repository ready at {REPO_DIR}")
subprocess.run(["ls", "-la", REPO_DIR])

## 2. Install dependencies

In [ ]:
req = os.path.join(REPO_DIR, "requirements.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req], check=True)
print("Dependencies installed.")

## 3. Mount Drive and configure model paths

Edit the paths below to match your Google Drive layout.

- `MULTICLASS_MODEL_DIR` must contain `multiclass_temporal_adapter_qwen3_instruct_improved.pt` and `multiclass_label_encoders.pkl`.
- `SUMMARIZATION_CHECKPOINT` is the Q-Former LoRA `checkpoint-400` directory.
- `BINARY_DRIVE_DIR` holds the binary weights (training-pipeline `instruct_*` naming); they are copied + renamed to what the app expects.
- `VIDEO_DIR` is a folder of sample videos.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# ---- MODEL PATHS (from training pipeline models_path.txt) ----
QWEN_MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
SUMMARIZATION_CHECKPOINT = "/content/drive/MyDrive/Project_VLM/ucf_qwen_v9_qformer/checkpoint-400"
MULTICLASS_MODEL_DIR = "/content/drive/MyDrive/models_ucf_v2/hpo_best_models"
BINARY_DRIVE_DIR = "/content/drive/MyDrive/models_ucf_v2"
VIDEO_DIR = "/content/drive/MyDrive/Project_VLM/All_Videos"
# --------------------------------------------------------------

# Copy + RENAME binary weights into the repo's model/ dir.
# binary_ensemble.py expects: input_dim.pkl, simple_adapter.pt, lora_model.pt
BINARY_MODEL_DIR = os.path.join(REPO_DIR, "model")
os.makedirs(BINARY_MODEL_DIR, exist_ok=True)

binary_file_map = {
    "instruct_input_dim.pkl": "input_dim.pkl",
    "instruct_simple_adapter.pt": "simple_adapter.pt",
    "lora_model.pt": "lora_model.pt",
}
for src_name, dst_name in binary_file_map.items():
    src = os.path.join(BINARY_DRIVE_DIR, src_name)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(BINARY_MODEL_DIR, dst_name))
        print(f"  Copied {src_name} -> {dst_name}")
    else:
        alt = os.path.join(BINARY_DRIVE_DIR, dst_name)
        if os.path.exists(alt):
            shutil.copy(alt, os.path.join(BINARY_MODEL_DIR, dst_name))
            print(f"  Copied {dst_name}")
        else:
            print(f"  MISSING {src_name} / {dst_name} (check BINARY_DRIVE_DIR)")

# Export env vars consumed by pipeline.py / binary_ensemble.py
os.environ["QWEN_MODEL_ID"] = QWEN_MODEL_ID
os.environ["SUMMARIZATION_CHECKPOINT"] = SUMMARIZATION_CHECKPOINT
os.environ["MULTICLASS_MODEL_DIR"] = MULTICLASS_MODEL_DIR
os.environ["BINARY_MODEL_DIR"] = BINARY_MODEL_DIR
os.environ["VIDEO_DIR"] = VIDEO_DIR
os.environ["PYTHONHASHSEED"] = "42"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

print("\nEnvironment configured:")
for k in ["QWEN_MODEL_ID", "SUMMARIZATION_CHECKPOINT", "MULTICLASS_MODEL_DIR", "BINARY_MODEL_DIR", "VIDEO_DIR"]:
    print(f"  {k} = {os.environ[k]}")

mc_ckpt = os.path.join(MULTICLASS_MODEL_DIR, "multiclass_temporal_adapter_qwen3_instruct_improved.pt")
mc_enc = os.path.join(MULTICLASS_MODEL_DIR, "multiclass_label_encoders.pkl")
print("\nFile checks:")
print(f"  multiclass ckpt : {'OK' if os.path.exists(mc_ckpt) else 'MISSING'}  {mc_ckpt}")
print(f"  multiclass enc  : {'OK' if os.path.exists(mc_enc) else 'MISSING'}  {mc_enc}")
print(f"  summary ckpt    : {'OK' if os.path.isdir(SUMMARIZATION_CHECKPOINT) else 'MISSING'}  {SUMMARIZATION_CHECKPOINT}")

## 4. Install the Cloudflare tunnel

In [ ]:
import urllib.request

CLOUDFLARED_BIN = "/usr/local/bin/cloudflared"

def _install_via_deb():
    deb = "/content/cloudflared.deb"
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"
    try:
        urllib.request.urlretrieve(url, deb)
        if os.path.getsize(deb) < 1_000_000:
            return False
        subprocess.run(["dpkg", "-i", deb], check=True)
        return bool(shutil.which("cloudflared"))
    except Exception as e:
        print(f"  .deb install failed: {e}")
        return False

def _install_via_binary():
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    for tool in (["wget", "-q", url, "-O", CLOUDFLARED_BIN],
                 ["curl", "-fL", url, "-o", CLOUDFLARED_BIN]):
        try:
            subprocess.run(tool, check=True)
            if os.path.getsize(CLOUDFLARED_BIN) > 1_000_000:
                subprocess.run(["chmod", "+x", CLOUDFLARED_BIN], check=True)
                return True
        except Exception as e:
            print(f"  {tool[0]} failed: {e}")
    try:
        urllib.request.urlretrieve(url, CLOUDFLARED_BIN)
        subprocess.run(["chmod", "+x", CLOUDFLARED_BIN], check=True)
        return os.path.getsize(CLOUDFLARED_BIN) > 1_000_000
    except Exception as e:
        print(f"  urllib failed: {e}")
        return False

if not (_install_via_deb() or _install_via_binary()):
    raise RuntimeError("Could not install cloudflared. Check Colab network and retry.")

CLOUDFLARED_BIN = shutil.which("cloudflared") or CLOUDFLARED_BIN
print("cloudflared at:", CLOUDFLARED_BIN)
subprocess.run([CLOUDFLARED_BIN, "--version"])

## 5. Launch Streamlit

Give it ~60s+ to boot and load the Qwen3-VL model. Auto-kills any stale process holding the port.

In [ ]:
PORT = 8501
LOG_PATH = "/content/streamlit.log"
app_path = os.path.join(REPO_DIR, "streamlit_app_unified.py")

subprocess.run(["pkill", "-f", "streamlit run"], check=False)
subprocess.run(["bash", "-c", f"fuser -k {PORT}/tcp 2>/dev/null || true"], check=False)
time.sleep(3)

streamlit_proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", app_path,
        "--server.port", str(PORT),
        "--server.address", "0.0.0.0",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--browser.gatherUsageStats", "false",
    ],
    stdout=open(LOG_PATH, "w"),
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

print("Waiting for Streamlit to start...")
for i in range(12):
    time.sleep(5)
    if streamlit_proc.poll() is not None:
        print("ERROR: Streamlit terminated. Recent log:")
        print(open(LOG_PATH).read()[-1500:])
        break
    print(f"  Still starting... ({(i+1)*5}s elapsed)")

print("\nRecent Streamlit log:")
print("".join(open(LOG_PATH).readlines()[-30:]))

## 6. Public tunnel

Verifies Streamlit is up, then opens a Cloudflare tunnel (retries up to 3x). If the browser shows NXDOMAIN, wait ~20-30s and refresh.

In [ ]:
import re as _re
import urllib.request as _urlreq

def _streamlit_up():
    try:
        with _urlreq.urlopen(f"http://localhost:{PORT}", timeout=5) as r:
            return r.status == 200
    except Exception:
        return False

print("Checking local Streamlit...")
for _ in range(24):
    if _streamlit_up():
        print("  Streamlit is up.")
        break
    if streamlit_proc.poll() is not None:
        print("ERROR: Streamlit died. Recent log:")
        print(open(LOG_PATH).read()[-1500:])
        raise SystemExit
    time.sleep(5)
else:
    print("  WARNING: not confirmed up; starting tunnel anyway.")

def _start_tunnel():
    proc = subprocess.Popen(
        [CLOUDFLARED_BIN, "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    url, start = None, time.time()
    while time.time() - start < 40:
        line = proc.stdout.readline()
        if not line:
            if proc.poll() is not None:
                break
            continue
        print(line.strip())
        m = _re.search(r"https://[\w-]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0)
            break
    return proc, url

tunnel_proc, public_url = None, None
for attempt in range(1, 4):
    print(f"\n--- Tunnel attempt {attempt} ---")
    tunnel_proc, public_url = _start_tunnel()
    if public_url:
        time.sleep(10)
        if tunnel_proc.poll() is None:
            print(f"\n*** PUBLIC URL: {public_url} ***")
            print("If NXDOMAIN, wait ~20-30s and refresh (DNS propagation).")
            break
        print("  Tunnel exited after URL; retrying...")
    else:
        print("  No URL captured; retrying...")
    try:
        tunnel_proc.kill()
    except Exception:
        pass

if not public_url:
    print("\nCould not establish a tunnel after 3 attempts. Re-run this cell.")